In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score, f1_score
from sklearn.impute import SimpleImputer

from data_preprocessing import create_train_test_val_sets, read_processed_data

In [2]:
#Create test train splits
x_mendeley, y_mendeley = read_processed_data(r"..\data\processed\mendeley_processed.csv")
x_phiusiil, y_phiusiil= read_processed_data(r"..\data\processed\phiusiil_processed.csv")

mendeley_sets = create_train_test_val_sets(x_mendeley,y_mendeley, label_col="Label", test_size=0.2, n_splits=5)
phiusiil_sets = create_train_test_val_sets(x_phiusiil,y_phiusiil, label_col="Label", test_size=0.2, n_splits=5)

Train/validation/test split prepared: 198360 instances for training and validation, 49590 instances for testing
Stratified 5-fold CV splits created.
Train/validation/test split prepared: 188636 instances for training and validation, 47159 instances for testing
Stratified 5-fold CV splits created.


In [3]:
def optimize_logistic_regression(X, y):

    params = {
        'model__C': [0.01, 0.1, 1, 10]
    }

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000))
    ])

    grid_search = GridSearchCV(
        pipeline,
        param_grid=params,
        cv=skf,
        scoring="f1",
        n_jobs=-1
    )

    grid_search.fit(X, y)

    print('\nBest hyperparameters:')
    print(grid_search.best_params_)

    return grid_search

In [4]:
def train_no_feature_selection(dataset, model):
    stratified_scores = []
    all_y_val = []
    all_y_pred = []

    for train_idx, val_idx in dataset["cv_splits"]:
        x_train = dataset["x_train_val"].iloc[train_idx]
        x_val = dataset["x_train_val"].iloc[val_idx]
        y_train = dataset["y_train_val"].iloc[train_idx]
        y_val = dataset["y_train_val"].iloc[val_idx]

        model.fit(x_train, y_train)
        y_pred = model.predict(x_val)

        stratified_scores.append(f1_score(y_val, y_pred))

        all_y_val.extend(y_val)
        all_y_pred.extend(y_pred)

    print(f"Mean F1: {np.mean(stratified_scores):.4f}, Std F1: {np.std(stratified_scores):.4f}")
    print("Validation Results:")
    print(classification_report(all_y_val, all_y_pred))

    # Final evaluation on hold-out test set
    model.fit(dataset["x_train_val"], dataset["y_train_val"])
    y_test_pred = model.predict(dataset["x_test"])

    print("\nTest Set Results:")
    print(classification_report(dataset["y_test"], y_test_pred))

In [5]:
print("Label in features?", "Label" in mendeley_sets["x_train_val"].columns)

print("Running hyperparameter tuning using Mendeley Dataset:")
logreg_mendeley = optimize_logistic_regression(
    mendeley_sets["x_train_val"],
    mendeley_sets["y_train_val"]
)

print("Running hyperparameter tuning using Phiusiil Dataset:")
logreg_phiusiil = optimize_logistic_regression(
    phiusiil_sets["x_train_val"],
    phiusiil_sets["y_train_val"]
)

print("Mendeley Results:")
train_no_feature_selection(mendeley_sets, logreg_mendeley.best_estimator_)

print("Phiusiil Results:")
train_no_feature_selection(phiusiil_sets, logreg_phiusiil.best_estimator_)

Label in features? False
Running hyperparameter tuning using Mendeley Dataset:


C:\Users\ovaze\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Best hyperparameters:
{'model__C': 0.1}
Running hyperparameter tuning using Phiusiil Dataset:


C:\Users\ovaze\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\model_selection\_validation.py:490: FitFailedWarning: 
7 fits failed out of a total of 12.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
2 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\ovaze\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ovaze\AppData\Local\Packages\PythonSoftwareFoundation


Best hyperparameters:
{'model__C': 0.01}
Mendeley Results:


...


Test Set Results:
              precision    recall  f1-score   support

           0       0.78      0.86      0.82     25708
           1       0.83      0.74      0.78     23882

    accuracy                           0.80     49590
   macro avg       0.81      0.80      0.80     49590
weighted avg       0.81      0.80      0.80     49590

Phiusiil Results:


C:\Users\ovaze\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\ovaze\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase t

Mean F1: 0.9993, Std F1: 0.0001
Validation Results:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     80756
           1       1.00      1.00      1.00    107880

    accuracy                           1.00    188636
   macro avg       1.00      1.00      1.00    188636
weighted avg       1.00      1.00      1.00    188636



C:\Users\ovaze\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Test Set Results:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     20189
           1       1.00      1.00      1.00     26970

    accuracy                           1.00     47159
   macro avg       1.00      1.00      1.00     47159
weighted avg       1.00      1.00      1.00     47159

